In [1]:
import xarray as xr
import glob
import os
import pandas as pd
from tqdm.notebook import tqdm

INPUT_DIR   = "/projekt1/ag_maahn/data_obs_nobackup/modis/MCD06COSP_D3/Amazon_cropped"
OUTPUT_FILE = "/projekt1/ag_maahn/data_obs_nobackup/modis/MCD06COSP_D3/merged/CPS_liq.nc"
GROUP = 'Cloud_Particle_Size_Liquid'

#choose Group: 'Cloud_Top_Pressure', 'Cloud_Mask_Fraction', 'Cloud_Mask_Fraction_Low', 'Cloud_Mask_Fraction_Mid', 'Cloud_Mask_Fraction_High', 'Cloud_Optical_Thickness_Liquid', 'Cloud_Optical_Thickness_Ice', 'Cloud_Optical_Thickness_Total', 'Cloud_Optical_Thickness_PCL_Liquid', 'Cloud_Optical_Thickness_PCL_Ice', 'Cloud_Optical_Thickness_PCL_Total', 'Cloud_Optical_Thickness_Log10_Liquid', 'Cloud_Optical_Thickness_Log10_Ice', 'Cloud_Optical_Thickness_Log10_Total', 'Cloud_Particle_Size_Liquid', 'Cloud_Particle_Size_Ice', 'Cloud_Particle_Size_PCL_Liquid', 'Cloud_Particle_Size_PCL_Ice', 'Cloud_Water_Path_Liquid', 'Cloud_Water_Path_Ice', 'Cloud_Water_Path_PCL_Liquid', 'Cloud_Water_Path_PCL_Ice', 'Cloud_Retrieval_Fraction_Liquid', 'Cloud_Retrieval_Fraction_Ice', 'Cloud_Retrieval_Fraction_Total', 'Cloud_Retrieval_Fraction_PCL_Liquid', 'Cloud_Retrieval_Fraction_PCL_Ice', 'Cloud_Retrieval_Fraction_PCL_Total'


files = sorted(glob.glob(os.path.join(INPUT_DIR, "**/*.nc")))
datasets = []

for filepath in tqdm(files, desc="Lade Dateien"):
    basename = os.path.basename(filepath)
    date_str = basename.split(".")[1]
    year = int(date_str[1:5])
    doy  = int(date_str[5:8])
    timestamp = pd.Timestamp(year, 1, 1) + pd.Timedelta(days=doy - 1)

    # Koordinaten aus Root laden
    root = xr.open_dataset(filepath)
    lat  = root.latitude.values
    lon  = root.longitude.values

    # Daten aus Gruppe laden
    ds = xr.open_dataset(filepath, group=GROUP)
    
    # Koordinaten zuweisen
    ds = ds.assign_coords(
        latitude=("latitude", lat),
        longitude=("longitude", lon)
    )
    ds = ds.expand_dims(time=[timestamp])
    datasets.append(ds)

print("Füge zusammen...")
merged = xr.concat(datasets, dim="time")

#first = xr.open_dataset(files[0], group=GROUP)
#merged = merged.assign_coords(
#    latitude=first.latitude.values,
#    longitude=first.longitude.values
#)

print("Speichere...")
merged.to_netcdf(OUTPUT_FILE)
print("Fertig!")

Lade Dateien:   0%|          | 0/8621 [00:00<?, ?it/s]

Füge zusammen...
Speichere...
Fertig!
